# Notebook 01 — WikiArt Data Preparation

**Purpose:** Download WikiArt from Kaggle, filter to the five target art styles,
resize all images to 256×256, and produce stratified train/val/test CSV splits
persisted on Google Drive.

**Cells:**
1. Setup — install packages, mount Drive, configure Kaggle credentials
2. Download WikiArt — pull dataset, inspect directory structure, print per-style counts
3. Preprocessing — resize/crop, save JPGs to Drive, create stratified splits
4. Verify — count per split/style, show 4×5 sample grid, print disk usage

**Resume safety:** All expensive outputs (processed images, split CSVs) land on
Google Drive.  Re-running any cell from the top is safe — existing files are
skipped via existence checks.

## Cell 1 — Setup
Install packages, mount Drive, and configure Kaggle API credentials.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install -q kaggle torch torchvision opencv-python albumentations \
               matplotlib scikit-learn tqdm pandas Pillow

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# ── Directory layout on Drive ────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive/art_restoration')
RAW_DIR      = Path('/content/wikiart_raw')          # Colab-local (fast SSD)
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'   # persisted on Drive
SPLIT_DIR    = DRIVE_ROOT / 'data' / 'splits'        # CSV split files
LOG_DIR      = DRIVE_ROOT / 'logs'

for d in [DRIVE_ROOT, PROCESSED_DIR, SPLIT_DIR, LOG_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Drive directories ready:')
for d in [DRIVE_ROOT, PROCESSED_DIR, SPLIT_DIR]:
    print(f'  {d}')

# ── Kaggle API credentials ────────────────────────────────────────────────────
# Option A: upload kaggle.json interactively (first run only)
# Option B: place kaggle.json in Drive at DRIVE_ROOT/kaggle.json beforehand

KAGGLE_CFG = Path.home() / '.kaggle'
KAGGLE_CFG.mkdir(exist_ok=True)
KAGGLE_KEY = KAGGLE_CFG / 'kaggle.json'

DRIVE_KAGGLE = DRIVE_ROOT / 'kaggle.json'
if DRIVE_KAGGLE.exists():
    shutil.copy(DRIVE_KAGGLE, KAGGLE_KEY)
    print('Kaggle credentials loaded from Drive.')
elif not KAGGLE_KEY.exists():
    from google.colab import files
    print('Upload your kaggle.json (from https://www.kaggle.com/account):')
    uploaded = files.upload()
    content = list(uploaded.values())[0]
    KAGGLE_KEY.write_bytes(content)
    # Also save to Drive for future runs
    DRIVE_KAGGLE.write_bytes(content)
    print('Credentials saved to Drive for future sessions.')

os.chmod(KAGGLE_KEY, 0o600)
print('Kaggle API ready.')

## Cell 2 — Download WikiArt from Kaggle
Download the dataset, detect its directory structure, filter to the five target
styles, and print per-style counts.

In [ ]:
import subprocess
from collections import Counter
from pathlib import Path

# ── Target styles (match exact directory names used in the dataset) ───────────
TARGET_STYLES: dict[str, str] = {
    # Kaggle directory name  →  canonical label used in splits
    'Renaissance':        'renaissance',
    'Baroque':            'baroque',
    'Impressionism':      'impressionism',
    'Post_Impressionism': 'post_impressionism',
    'Realism':            'realism',
    # Some mirrors use lowercase or underscores — covered below
    'renaissance':        'renaissance',
    'baroque':            'baroque',
    'impressionism':      'impressionism',
    'post_impressionism': 'post_impressionism',
    'realism':            'realism',
    'Post-Impressionism': 'post_impressionism',
    'post-impressionism': 'post_impressionism',
}

RAW_DIR = Path('/content/wikiart_raw')

def _download_dataset(slug: str, dest: Path) -> bool:
    """Try to download a Kaggle dataset. Returns True on success."""
    print(f'Trying dataset: {slug} ...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', slug, '--unzip', '-p', str(dest)],
        capture_output=True, text=True,
    )
    print(result.stdout[-2000:] if result.stdout else '')  # last 2 KB
    if result.returncode != 0:
        print(f'  FAILED: {result.stderr[:500]}')
        return False
    return True

# ── Download (try primary then mirrors) ────────────────────────────────────────
datasets_to_try = [
    'ipythonx/wikiart',
    'steubk/wikiart',
    'antoinegruson/-wikiart',
]

# Only download if raw dir is empty
existing_imgs = list(RAW_DIR.rglob('*.jpg')) + list(RAW_DIR.rglob('*.png'))
if existing_imgs:
    print(f'Raw images already present ({len(existing_imgs):,}). Skipping download.')
else:
    for slug in datasets_to_try:
        if _download_dataset(slug, RAW_DIR):
            break
    else:
        raise RuntimeError(
            'All dataset mirrors failed. Check your Kaggle credentials '
            'and network access, or manually place images under /content/wikiart_raw.'
        )

# ── Discover directory structure ──────────────────────────────────────────────
print('\nTop-level directories in raw download:')
top_dirs = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
for d in top_dirs[:30]:
    n = len(list(d.glob('*.jpg'))) + len(list(d.glob('*.png')))
    print(f'  {d.name:40s}  {n:5d} images')

# ── Collect target-style paths ────────────────────────────────────────────────
all_images: list[tuple[Path, str]] = []  # (path, canonical_style)

def _collect_from_dir(root: Path) -> None:
    for style_dir in root.iterdir():
        if not style_dir.is_dir():
            continue
        canonical = TARGET_STYLES.get(style_dir.name)
        if canonical is None:
            continue
        imgs = list(style_dir.glob('*.jpg')) + list(style_dir.glob('*.png'))
        all_images.extend((p, canonical) for p in imgs)

_collect_from_dir(RAW_DIR)
# Some mirrors nest inside a subdirectory
for sub in RAW_DIR.iterdir():
    if sub.is_dir() and len(all_images) == 0:
        _collect_from_dir(sub)

# ── Statistics ────────────────────────────────────────────────────────────────
style_counts = Counter(style for _, style in all_images)
print(f'\nFiltered to {len(style_counts)} target styles:')
for style, count in sorted(style_counts.items()):
    print(f'  {style:25s} {count:5d} images')
print(f'\nTotal: {len(all_images):,} paintings')

if len(all_images) < 1000:
    print('WARNING: fewer than 1,000 images found — check dataset slug or directory names.')
elif len(all_images) >= 5000:
    print('Target of 5,000+ images reached. ✓')
else:
    print(f'Note: {len(all_images):,} images found (target ≥ 5,000 — consider augmentation or extra styles).')

## Cell 3 — Preprocessing
Resize each painting to 256×256 (aspect-preserving center crop), save as JPG
quality=95 to Google Drive, and produce stratified 80/10/10 split CSVs.

In [ ]:
import hashlib
import random
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

SEED = 42
IMAGE_SIZE = 256
JPG_QUALITY = 95

PROCESSED_DIR = Path('/content/drive/MyDrive/art_restoration/data/processed')
SPLIT_DIR     = Path('/content/drive/MyDrive/art_restoration/data/splits')


def resize_center_crop(img: Image.Image, size: int) -> Image.Image:
    """Aspect-preserving resize so the short side equals *size*, then center-crop."""
    w, h = img.size
    scale = size / min(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    img = img.resize((new_w, new_h), Image.LANCZOS)
    left = (new_w - size) // 2
    top  = (new_h - size) // 2
    return img.crop((left, top, left + size, top + size))


def stable_hash(path: Path) -> str:
    """Short deterministic identifier derived from the file path."""
    return hashlib.md5(path.name.encode()).hexdigest()[:8]


# ── Process images (resume-safe: skip already-converted files) ────────────────
records: list[dict[str, Any]] = []
skipped = 0
errors  = 0

for src_path, style in tqdm(all_images, desc='Preprocessing'):
    dst_name = f"{style}_{stable_hash(src_path)}.jpg"
    dst_path = PROCESSED_DIR / style / dst_name
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    if dst_path.exists():
        skipped += 1
    else:
        try:
            with Image.open(src_path) as img:
                img = img.convert('RGB')
                img = resize_center_crop(img, IMAGE_SIZE)
                img.save(dst_path, 'JPEG', quality=JPG_QUALITY, optimize=True)
        except Exception as exc:
            print(f'  Error processing {src_path.name}: {exc}')
            errors += 1
            continue

    records.append({'path': str(dst_path), 'style': style})

print(f'\nProcessed : {len(records) - skipped:,}')
print(f'Skipped   : {skipped:,}  (already on Drive)')
print(f'Errors    : {errors:,}')

# ── Stratified 80 / 10 / 10 split ─────────────────────────────────────────────
df = pd.DataFrame(records)

train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df['style'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['style'], random_state=SEED
)

train_df['split'] = 'train'
val_df['split']   = 'val'
test_df['split']  = 'test'

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
full_df.to_csv(SPLIT_DIR / 'all_splits.csv', index=False)
train_df.to_csv(SPLIT_DIR / 'train.csv', index=False)
val_df.to_csv(SPLIT_DIR / 'val.csv', index=False)
test_df.to_csv(SPLIT_DIR / 'test.csv', index=False)

print(f'\nSplit CSVs saved to {SPLIT_DIR}')
print(f'  train : {len(train_df):,}')
print(f'  val   : {len(val_df):,}')
print(f'  test  : {len(test_df):,}')

## Cell 4 — Verify
Print per-split per-style counts, display a 4×5 sample grid with style labels,
and report total disk usage.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

SPLIT_DIR     = Path('/content/drive/MyDrive/art_restoration/data/splits')
PROCESSED_DIR = Path('/content/drive/MyDrive/art_restoration/data/processed')

full_df = pd.read_csv(SPLIT_DIR / 'all_splits.csv')

# ── Per-split per-style table ─────────────────────────────────────────────────
pivot = (
    full_df.groupby(['style', 'split'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['train', 'val', 'test'])
)
pivot['total'] = pivot.sum(axis=1)
pivot.loc['TOTAL'] = pivot.sum()
print('Per-style counts per split:')
print(pivot.to_string())

# ── 4×5 sample grid ───────────────────────────────────────────────────────────
ROWS, COLS = 4, 5
sample_rows = full_df[full_df['split'] == 'train'].sample(ROWS * COLS, random_state=42)

fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 2.5, ROWS * 2.5))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img = Image.open(row['path'])
    ax.imshow(np.asarray(img))
    ax.set_title(row['style'].replace('_', ' ').title(), fontsize=7)
    ax.axis('off')

plt.suptitle('Random train samples (256×256)', fontsize=11)
plt.tight_layout()
plt.show()

# ── Disk usage ────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['du', '-sh', str(PROCESSED_DIR)],
    capture_output=True, text=True,
)
disk_usage = result.stdout.split()[0] if result.returncode == 0 else 'N/A'

print(f'\nProcessed dataset size on Drive: {disk_usage}')
print(f'Total images in splits        : {len(full_df):,}')
print('Data preparation complete. ✓')